# Wizualizacja sekwencji, GT i DET

Poniżej są trzy komórki:
1) Odtwarzanie sekwencji z folderu `img1`.
2) Odtwarzanie sekwencji z naniesionymi bounding boxami z `gt.txt`.
3) Odtwarzanie sekwencji z naniesionymi bounding boxami z `det.txt`.

In [1]:
import os
import glob
import cv2
from collections import defaultdict

### Wizualizacja sekwencji z folderu `img1`

In [ ]:
# Zmienna z nazwa sekwencji (np. MOT_03/MOT_04/MOT_05)
seq_name = "MOT_02"

# Zmienna nazwa zbioru (evs_mot-train/evs_mot-test)
dataset_name = "evs_mot-train"

base_dir = os.path.join("..", "..", "..", "proj02", "evs_mot_public_dataset", dataset_name, seq_name)
img_dir = os.path.join(base_dir, "img1")

image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

for path in image_paths:
    frame = cv2.imread(path)
    if frame is None:
        continue

    frame_number = int(os.path.splitext(os.path.basename(path))[0])
    cv2.putText(frame, f"Frame: {frame_number}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

    cv2.imshow("Sequence", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == 27 or key == ord("q"):
        break

cv2.destroyAllWindows()

### Wizualizacja sekwencji z naniesionymi bounding boxami z `gt.txt`.

In [2]:
# Zmienna z nazwa sekwencji (np. MOT_03/MOT_04/MOT_05)
seq_name = "MOT_02"

base_dir = os.path.join("..", "..", "..", "proj02", "evs_mot_public_dataset", "evs_mot-train", seq_name)
img_dir = os.path.join(base_dir, "img1")
gt_path = os.path.join(base_dir, "gt", "gt.txt")

# Wczytaj GT: frame,id,bb_left,bb_top,bb_width,bb_height,eval_flag,class,visibility
boxes_by_frame = defaultdict(list)

with open(gt_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(",")
        if len(parts) < 9:
            continue

        frame_id = int(float(parts[0]))
        obj_id = int(float(parts[1]))
        bb_left = float(parts[2])
        bb_top = float(parts[3])
        bb_width = float(parts[4])
        bb_height = float(parts[5])
        eval_flag = int(float(parts[6]))
        class_id = int(float(parts[7]))

        # Opcjonalna filtracja: eval_flag==1 i class_id==1 (ludzie)
        if eval_flag != 1 or class_id != 1:
            continue

        x1 = int(round(bb_left))
        y1 = int(round(bb_top))
        x2 = int(round(bb_left + bb_width))
        y2 = int(round(bb_top + bb_height))

        boxes_by_frame[frame_id].append((x1, y1, x2, y2, obj_id))


def color_for_id(obj_id: int) -> tuple:
    return (
        int((obj_id * 37) % 256),
        int((obj_id * 17) % 256),
        int((obj_id * 29) % 256),
    )


image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

for path in image_paths:
    frame = cv2.imread(path)
    if frame is None:
        continue

    # Numer klatki z nazwy pliku, np. 000001.jpg -> 1
    frame_number = int(os.path.splitext(os.path.basename(path))[0])

    for (x1, y1, x2, y2, obj_id) in boxes_by_frame.get(frame_number, []):
        color = color_for_id(obj_id)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text_pos = (x1, max(0, y1 - 5))
        cv2.putText(frame, str(obj_id), text_pos, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.putText(frame, f"Frame: {frame_number}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

    cv2.imshow("Sequence + GT", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == 27 or key == ord("q"):
        break

cv2.destroyAllWindows()

### Wizualizacja sekwencji z naniesionymi bounding boxami z `det.txt`.

In [ ]:
# Zmienna z nazwa sekwencji (np. MOT_03/MOT_04/MOT_05)
seq_name = "MOT_02"

# Zmienna nazwa zbioru (evs_mot-train/evs_mot-test)
dataset_name = "evs_mot-train"

base_dir = os.path.join("..", "..", "..", "proj02", "evs_mot_public_dataset", dataset_name, seq_name)
img_dir = os.path.join(base_dir, "img1")
det_path = os.path.join(base_dir, "det", "det.txt")

# Wczytaj det: frame,-1,bb_left,bb_top,bb_width,bb_height,detection_confidence
boxes_by_frame = defaultdict(list)

with open(det_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(",")
        if len(parts) < 7:
            continue

        frame_id = int(float(parts[0]))
        bb_left = float(parts[2])
        bb_top = float(parts[3])
        bb_width = float(parts[4])
        bb_height = float(parts[5])

        x1 = int(round(bb_left))
        y1 = int(round(bb_top))
        x2 = int(round(bb_left + bb_width))
        y2 = int(round(bb_top + bb_height))

        boxes_by_frame[frame_id].append((x1, y1, x2, y2))

image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

for path in image_paths:
    frame = cv2.imread(path)
    if frame is None:
        continue

    # Numer klatki z nazwy pliku, np. 000001.jpg -> 1
    frame_number = int(os.path.splitext(os.path.basename(path))[0])

    for (x1, y1, x2, y2) in boxes_by_frame.get(frame_number, []):
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 128, 255), 2)

    cv2.putText(frame, f"Frame: {frame_number}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

    cv2.imshow("Sequence + DET", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == 27 or key == ord("q"):
        break

cv2.destroyAllWindows()